In [1]:
import jax.numpy as jnp
import numpy as np
from scipy.stats import invwishart

In [ ]:
def generate_mixture_simulated_data(n_units=300, n_obs=50, n_alts=4, n_components=2, 
                                    n_params=None, n_demos=2, custom_pvec=None, 
                                    custom_indicators=None, seed=42):
    """
    Generate synthetic panel data for a Hierarchical Multinomial Logit (HMNL) model.

    This function simulates choice data where heterogeneous decision-makers (units)
    choose among mutually exclusive alternatives over multiple observation periods.
    Unit-level preferences are drawn from a finite mixture of multivariate normal
    distributions. The means of these components are shifted by observed,
    mean-centered demographic characteristics (based on Rossi et al., 2006).

    The data generation process follows the same distributions as used for priors:
    - Mixture probabilities (pvec) ~ Dirichlet(alpha)
    - Covariance matrices   (Sigma_k) ~ InverseWishart(nu, V)
    - Component means       (mu_k) ~ Normal(mu_bar, Sigma_k / a_mu)

    Args:
        n_units (int): Number of decision-making units (e.g., consumers). Defaults to 300.
        n_obs (int): Number of choice occasions per unit. Defaults to 50.
        n_alts (int): Number of mutually exclusive alternatives per choice. Defaults to 4.
        n_components (int): Number of latent segments (K) in the mixture. Defaults to 2.
        n_params (int, optional): Total parameters in the utility function. Defaults to None,
            which sets it to `n_alts` (allows n_alts - 1 constants + 1 continuous var).
        n_demos (int): Number of observable demographic characteristics. Defaults to 2.
        custom_pvec (array-like, optional): User-defined probabilities for latent classes.
            Must sum to 1 and be of length `n_components`. Defaults to None.
        custom_indicators (array-like, optional): User-defined latent class assignments
            for each unit. Must be of length `n_units`. Defaults to None.
        seed (int): Random seed for reproducibility. Defaults to 42.

    Returns:
        dict: A dictionary containing the simulated choice data, design matrices,
            demographics, and all true underlying generative parameters (cast to JAX arrays
            where applicable for downstream modeling).
            
            Keys:
            - 'X': JAX array of design matrices, shape (N*T, n_alts, n_params).
            - 'y': JAX array of observed choices, shape (N*T,).
            - 'Z': JAX array of demographics, shape (n_units, n_demos).
            - 'unit_idx': JAX array mapping observations to units, shape (N*T,).
            - 'n_units', 'n_params', 'n_demos', 'K', 'n_alts': Dimension integers.
            - 'TRUE_DELTA': True demographic shift matrix.
            - 'TRUE_BETA': True unit-level parameters.
            - 'TRUE_PVEC': True mixture probabilities.
            - 'TRUE_MU_K': True mixture component means.
            - 'TRUE_SIGMA_K': True mixture component covariance matrices.
            - 'TRUE_INDICATORS': True latent class assignments.

    Raises:
        ValueError: If `n_params` < `n_alts - 1`.
        ValueError: If `custom_pvec` length does not match `n_components`.
        ValueError: If `custom_indicators` length does not match `n_units`.
    """
    np.random.seed(seed)
    
    # ==========================================================================
    # Handle Dimensions & Parameter Initialization
    # ==========================================================================
    # Default behavior: Ensure enough parameters exist to identify the base
    # intercepts (ASCs) for the alternatives plus at least one continuous feature.
    if n_params is None:
        n_params = n_alts 
        
    if n_params < n_alts - 1:
        raise ValueError(
            f"n_params ({n_params}) must be at least n_alts - 1 "
            f"({n_alts - 1}) to allow for Alternative Specific Constants (ASCs)."
        )

    # Number of ASCs is always alternatives minus 1 (one base alternative)
    n_ascs = n_alts - 1
    # Remaining parameters are treated as continuous variables (e.g., price)
    n_continuous = n_params - n_ascs

    # ==========================================================================
    # Generate Demographic Data (Z)
    # ==========================================================================
    # Draw standard normal demographics for each unit
    Z = np.random.normal(0, 1, size=(n_units, n_demos))
    
    # Mean-centering constraint: Z must be centered so that the mean of the 
    # unit-level parameters (theta/beta) is entirely determined by the normal 
    # mixture component means (mu_k).
    Z = Z - np.mean(Z, axis=0) 

    # ==========================================================================
    # Generate TRUE Global & Mixture Parameters (Conjugate Priors)
    # ==========================================================================
    # Delta governs how demographics shift the baseline utilities
    Delta_true = np.random.normal(0, 0.5, size=(n_demos, n_params))

    # --- 3a. Mixture Probabilities (pvec) ~ Dirichlet(alpha) ---
    if custom_pvec is not None:
        if len(custom_pvec) != n_components:
            raise ValueError(
                f"custom_pvec length ({len(custom_pvec)}) must match "
                f"n_components ({n_components})."
            )
        true_pvec = np.array(custom_pvec)
        true_pvec = true_pvec / np.sum(true_pvec) # Normalize to guarantee sum=1
    else:
        # Use a flat, symmetric Dirichlet prior for the mixture components
        alpha_prior = np.ones(n_components) * 2.0 
        true_pvec = np.random.dirichlet(alpha_prior)

    # --- Covariance Matrices (Sigma_k) ~ InverseWishart(nu, V) ---
    # Degrees of freedom (nu) must strictly exceed n_params - 1 for a proper prior
    nu = n_params + 2 
    V = np.eye(n_params) # Base scale matrix (Identity)
    
    true_Sigma_k = np.zeros((n_components, n_params, n_params))
    true_mu_k = np.zeros((n_components, n_params))
    
    # Prior hyperparameters for the component means (mu_k)
    mu_bar = np.zeros(n_params)
    a_mu = 1.0 # Scaling factor for the conditional variance of mu_k

    for k in range(n_components):
        # Draw a full, positive-definite covariance matrix for latent segment k
        true_Sigma_k[k] = invwishart.rvs(df=nu, scale=V)
        
        # --- Component Means (mu_k) ~ N(mu_bar, Sigma_k * a_mu^-1) ---
        # Draw the component mean conditionally based on its generated covariance
        true_mu_k[k] = np.random.multivariate_normal(
            mu_bar, 
            true_Sigma_k[k] / a_mu
        )

    # ==========================================================================
    # Generate Individual-Level Parameters (Heterogeneity Equation)
    # ==========================================================================
    beta_true = np.zeros((n_units, n_params))
    
    # Assign latent classes to units (either custom or multinomial draws)
    if custom_indicators is not None:
        if len(custom_indicators) != n_units:
            raise ValueError(
                f"custom_indicators length ({len(custom_indicators)}) must "
                f"match n_units ({n_units})."
            )
        true_indicators = np.array(custom_indicators)
    else:
        # ind_i ~ Multinomial(pvec)
        true_indicators = np.random.choice(n_components, size=n_units, p=true_pvec)

    for i in range(n_units):
        k = true_indicators[i] # Retrieve unit i's assigned latent class
        
        # Calculate the unit-specific mean: demographic shift + baseline component mean
        mu_i = Z[i] @ Delta_true + true_mu_k[k]
        
        # Draw the final parameter vector for unit i
        beta_true[i] = np.random.multivariate_normal(mu_i, true_Sigma_k[k])

    # ==========================================================================
    # Generate Choice Data (X and y) via Multinomial Logit (MNL)
    # ==========================================================================
    X_list, y_list, unit_idx_list = [], [], []

    # Iterate through panels: each unit makes 'n_obs' choices
    for i in range(n_units):
        for t in range(n_obs):
            # Initialize design matrix for occasion t
            X_it = np.zeros((n_alts, n_params))

            # Populate Alternative Specific Constants (ASCs)
            # Alternative 0 is the reference (all zeros). Others get dummy variables.
            for a in range(1, n_alts):
                X_it[a, a - 1] = 1.0

            # Populate continuous variables (e.g., random prices for each alternative)
            if n_continuous > 0:
                X_it[:, n_ascs:] = np.random.uniform(
                    1.0, 5.0, size=(n_alts, n_continuous)
                )

            # Calculate deterministic utility vector for all alternatives: U = X * beta
            U_it = X_it @ beta_true[i]
            
            # Softmax transformation to probabilities (using max-subtraction trick 
            # to prevent numeric overflow during exponentiation)
            U_it_max = np.max(U_it)
            exp_U = np.exp(U_it - U_it_max)
            probs = exp_U / np.sum(exp_U)

            # Simulate the observed categorical choice based on calculated probabilities
            y_it = np.random.choice(n_alts, p=probs)

            # Append to sequential tracking lists
            X_list.append(X_it)
            y_list.append(y_it)
            unit_idx_list.append(i)

    # ==========================================================================
    # Return Packaged Data
    # ==========================================================================
    # Convert data matrices to JAX arrays to optimize downstream tensor operations
    return {
        "X": jnp.array(X_list),
        "y": jnp.array(y_list),
        "Z": jnp.array(Z),
        "unit_idx": jnp.array(unit_idx_list),
        "n_units": n_units,
        "n_params": n_params,
        "n_demos": n_demos,
        "K": n_components,
        "n_alts": n_alts,
        "TRUE_DELTA": Delta_true,
        "TRUE_BETA": beta_true,
        "TRUE_PVEC": true_pvec,
        "TRUE_MU_K": true_mu_k,
        "TRUE_SIGMA_K": true_Sigma_k,
        "TRUE_INDICATORS": true_indicators
    }

In [4]:
import pandas as pd
import numpy as np

print("Running simulation...")

# Generate a minimal dataset: 5 units, 3 observations each, 3 alternatives
sim_data = generate_mixture_simulated_data(
    n_units=5,      
    n_obs=3,        
    n_alts=3,       
    n_components=2, 
    n_params=4,     # 2 Alternative Specific Constants + 2 Continuous Variables
    n_demos=2,      
    seed=123
)

print("\n--- Simulation Complete ---")

# 1. Inspect Demographics (Z)
print("\n1. Mean-Centered Demographics (Z) for the 5 units:")
z_df = pd.DataFrame(np.array(sim_data["Z"]), columns=["Demo_1", "Demo_2"])
z_df.index.name = "Unit_ID"
print(z_df.round(3))

# 2. Inspect Latent Classes and Unit Parameters (Beta)
print("\n2. True Parameters (Beta) and Assigned Latent Classes:")
beta_cols = [f"Beta_{i}" for i in range(sim_data["n_params"])]
beta_df = pd.DataFrame(np.array(sim_data["TRUE_BETA"]), columns=beta_cols)
beta_df["Latent_Class (k)"] = np.array(sim_data["TRUE_INDICATORS"])
beta_df.index.name = "Unit_ID"
print(beta_df.round(3))

# 3. Inspect a Single Choice Occasion
print("\n3. First Choice Occasion (Unit 0, Obs 0):")

# The design matrix for the first observation
x_sample = np.array(sim_data['X'][0])
x_cols = ["ASC_1", "ASC_2", "Cont_Var_1", "Cont_Var_2"]
x_df = pd.DataFrame(x_sample, columns=x_cols)
x_df.index = [f"Alternative {i}" for i in range(sim_data["n_alts"])]

print("\nDesign Matrix (X):")
print(x_df.round(3))

# The actual choice made by the unit for this occasion
print(f"\nResulting Choice (y): Alternative {sim_data['y'][0]}")

Running simulation...

--- Simulation Complete ---

1. Mean-Centered Demographics (Z) for the 5 units:
         Demo_1  Demo_2
Unit_ID                
0        -0.577   1.028
1         0.791  -1.476
2        -0.070   1.682
3        -1.918  -0.398
4         1.774  -0.836

2. True Parameters (Beta) and Assigned Latent Classes:
         Beta_0  Beta_1  Beta_2  Beta_3  Latent_Class (k)
Unit_ID                                                  
0        -0.424  -0.789  -0.108   1.233                 1
1        -0.422  -0.701  -2.531  -2.050                 0
2        -0.742  -1.951   1.411   1.931                 1
3         0.118  -0.192  -1.695  -0.018                 1
4        -1.207  -0.744  -0.422  -1.652                 1

3. First Choice Occasion (Unit 0, Obs 0):

Design Matrix (X):
               ASC_1  ASC_2  Cont_Var_1  Cont_Var_2
Alternative 0    0.0    0.0       1.174       2.219
Alternative 1    1.0    0.0       2.593       3.820
Alternative 2    0.0    1.0       4.981       2.